## MLflow Prompt Evaluation

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key

### Instantiating the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Enable MLflow Tracing

In [ ]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("ticket-classification-evaluation")
mlflow.set_tracking_uri("http://localhost:5000")

### Create Prompt Version 1

In [ ]:
PROMPT_NAME = "ticket-classification-prompt"

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)

print(f"Created Version: {prompt_v1.version}")

### Create Improved Prompt Version 2

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
You are an enterprise IT support ticket classifier.

Analyze the following support ticket:

{{ticket}}

Classify the ticket using ONLY one of these categories:

- Network
- Authentication
- Hardware
- Software
- Security

Assign ONLY one of these priority levels:

- Low
- Medium
- High
- Critical

Priority Guidelines:

Critical:
Major security incident or widespread business outage.

High:
A serious issue preventing a user or team from working.

Medium:
An issue affecting productivity but with a workaround available.

Low:
Minor issue or informational request.

Return ONLY the following format:

Category: 
Priority: 
""",

    commit_message=(
        "v2: Added classification taxonomy, "
        "priority rules and output constraints"
    )
)

print(f"Created Version: {prompt_v2.version}")

### Create the Evaluation Dataset

In [ ]:
eval_dataset = mlflow.genai.datasets.create_dataset(
    name = "ticket-classification-eval-dataset",
)

In [ ]:
evaluation_examples = [

    {
        "inputs": {
            "ticket":
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
        },
        "expectations": {
            "expected_category": "Authentication",
            "expected_priority": "High"
        }
    },

    {
        "inputs": {
            "ticket":
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
        },
        "expectations": {
            "expected_category": "Network",
            "expected_priority": "Critical"
        }
    },

    {
        "inputs": {
            "ticket":
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
        },
        "expectations": {
            "expected_category": "Software",
            "expected_priority": "Medium"
        }
    },

    {
        "inputs": {
            "ticket":
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
        },
        "expectations": {
            "expected_category": "Security",
            "expected_priority": "High"
        }
    },

    {
        "inputs": {
            "ticket":
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
        },
        "expectations": {
            "expected_category": "Hardware",
            "expected_priority": "Low"
        }
    }
]

In [ ]:
eval_dataset = eval_dataset.merge_records(
    evaluation_examples
)

print(
    f"Added {len(evaluation_examples)} "
    "evaluation records."
)

### Create the Classification Function

In [ ]:
from langchain_core.messages import HumanMessage

def create_classifier(prompt_name, version):

    @mlflow.trace
    def classify_ticket(ticket: str):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{prompt_name}/{version}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=ticket
        )

        response = model.invoke(
            [
                HumanMessage(content=formatted_prompt)
            ]
        )

        return response.content

    return classify_ticket

### Create a Classification Judge

In [ ]:
from mlflow.genai import make_judge

classification_judge = make_judge(

    name="classification_accuracy",

    instructions="""
Evaluate whether the generated IT support ticket classification
matches the expected classification.

Generated classification:
{{ outputs }}

Expected classification:
{{ expectations }}

The expectations contain:
- expected_category: the correct ticket category
- expected_priority: the correct ticket priority

Return true ONLY if BOTH:
1. The generated category matches expected_category.
2. The generated priority matches expected_priority.

Otherwise return false.
""",

    feedback_value_type=bool,

    model="anthropic:/claude-opus-4-5"
)

### Create an Output Format Judge

In [ ]:
format_judge = make_judge(

    name="output_format_compliance",

    instructions="""
Evaluate whether the generated output follows exactly
this structure:

Category: 
Priority: 

The category must be one of:
Network, Authentication, Hardware, Software, Security

The priority must be one of:
Low, Medium, High, Critical

Generated output:

{{ outputs }}

Return true if the output follows these requirements.
Otherwise return false.
""",

    feedback_value_type=bool,

    model = "anthropic:/claude-opus-4-5"
)

### Evaluate Both the Prompts

In [ ]:
scorers = [
    classification_judge,
    format_judge
]

results = {}

for version in [1, 2]:

    print(f"\nEvaluating Version {version}...")

    with mlflow.start_run(
        run_name=f"ticket_classifier_v{version}"
    ):

        mlflow.log_param(
            "prompt_version",
            version
        )

        eval_results = mlflow.genai.evaluate(

            predict_fn=create_classifier(
                PROMPT_NAME,
                version
            ),

            data=eval_dataset,

            scorers=scorers
        )

        results[f"v{version}"] = eval_results

### Compare the Metrics

In [ ]:
ACCURACY_METRIC = "classification_accuracy/mean"
FORMAT_METRIC = "output_format_compliance/mean"


def get_metric(result, metric):

    if metric not in result.metrics:
        raise KeyError(
            f"{metric} not found. "
            f"Available metrics: {result.metrics.keys()}"
        )

    return result.metrics[metric]

In [ ]:
print("===== PROMPT COMPARISON =====")

for version, result in results.items():

    accuracy = get_metric(
        result,
        ACCURACY_METRIC
    )

    format_score = get_metric(
        result,
        FORMAT_METRIC
    )

    print(f"\n{version}")

    print(
        f"Classification Accuracy: {accuracy:.2f}"
    )

    print(
        f"Format Compliance: {format_score:.2f}"
    )